# v5 — LoRA vs full fine-tuning
Same data and settings as v3; only the method changes. LoRA trains ~1.5% of the weights.
Note LoRA does not shrink deployment: the adapter is merged back into a normal 249 MB model.
Runtime → T4 GPU → Run all. ~1.2 h.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v5_lora_finetuning'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v5_lora_finetuning'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v5_lora_finetuning/results'
!pip -q install -U transformers datasets accelerate scikit-learn peft
# Colab ships torchao 0.10, which peft rejects while scanning layer types; we do not use it
!pip uninstall -y -q torchao 2>/dev/null; echo removed torchao

In [ ]:
!python prepare_data.py --classes 5 --per-class 40000

In [ ]:
# ~5 min: 100 steps each way, purely to measure peak GPU memory and trainable parameters
!python finetune_roberta.py --name memory_full_finetune --data-dir data5 --batch-size 32 --limit 3200 --epochs 1 --eval-steps 100 --results-dir {RESULTS} --ckpt-dir /content/memory_full_finetune
!python finetune_roberta.py --name memory_lora --data-dir data5 --batch-size 32 --limit 3200 --epochs 1 --eval-steps 100 --lora --results-dir {RESULTS} --ckpt-dir /content/memory_lora
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ('memory_full_finetune', 'memory_lora'):
    print(f"{k}: peak GPU {m[k].get('peak_gpu_gb')} GB, trainable params {m[k].get('trainable_params'):,}")

In [ ]:
# ~50-60 min
!python finetune_roberta.py --name roberta_base_5class_lora --data-dir data5 --batch-size 32 --eval-steps 2500 --lora --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_base_5class_lora/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['roberta_base_5class_lora', 'roberta_base_5class_lora_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!zip -qr btp_v5_lora_finetuning.zip models/roberta_base_5class_lora versions/v5_lora_finetuning/results && ls -lh btp_v5_lora_finetuning.zip
!cp btp_v5_lora_finetuning.zip {DRIVE}/
from google.colab import files
files.download('btp_v5_lora_finetuning.zip')